# Tarea 2
Integrantes: Claudio Alvial, Cristóbal Navarro, Cristopher Vasques.


## Parte 1

## Creación de CNN propia
Primero inst

## Parte 2

### Faster C-NN
___

In [2]:
import os
from pathlib import Path

import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_pil_image
from torchvision.io import read_image
import pandas as pd

#=== Variables varias ===
CARPETA_ENTRADA = "data/FjordFish/images/train"
CARPETA_SALIDA  = Path("resultados")      
ARCHIVO_CSV     = "detecciones.csv" 
UMBRAL_CONFIANZA    = 0.9         
EXTENSION = ".jpg" 


# Creamos la carpeta de salida de las imagenes procesadas
os.makedirs(CARPETA_SALIDA, exist_ok=True)


#=== Cargamos el modelo faster rc-nn, en su version 2 (EXPLICAR EL PQ USAMOS LA VERSION 2 IMPORTANTE) ===

#Usamos los pesos del faster rc-nn que usa los pesos entrenados del dataset COCO :D
pesos = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(
    #Pasamos los pesos al modelo
    weights=pesos,

    #Pasamos el umbral de confianza, con el cual obtendrá las cajas que sobrepasen dicho umbral 
    box_score_thresh=UMBRAL_CONFIANZA,

).to("cpu")
model.eval()

#Obtenemos la transformación de la imagen usada en el entrenamiento
preprocess = pesos.transforms()

#=== Guardamos las rutas de las imagenes ===
carpeta = Path(CARPETA_ENTRADA)
rutas = []

# Guardamos las rutas de las imagenes y luego las ordenamos de menor a mayor por su numero de foto
for archivo in sorted(carpeta.iterdir()):
    if archivo.suffix == EXTENSION:
        rutas.append(archivo)

#Función para obtener el numero de cada foto
def extraer_numero(path):
    numeros = re.findall(r'\d+', path.name)
    if numeros != 0:
        return int(numeros[0])
    else:
        return 0


#Ordenamos las rutas por el numero de foto
rutas.sort(key=extraer_numero)
print(f"Se encontraron {len(rutas)} imágenes en '{CARPETA_ENTRADA}'")



#=== procesamos las imagenes y guardamos su información ===

#En esta lista van guardados los objetos encontrados, junto con su información más relevante
registros = []  

for ruta in rutas:
    print(f"Procesando: {ruta.name}")
    # leemos la imagen
    img = read_image(str(ruta))

    # Preprocesar y mover al dispositivo
    batch = [preprocess(img).to("cpu")]

    # Desactivamos el calculo de la gradiente para adivinar y no entrenar
    with torch.no_grad():
        prediction = model(batch)[0]


    cajas  = prediction["boxes"].cpu()
    etiqueta = prediction["labels"].cpu()
    confianza = prediction["scores"].cpu()

    # Caso para cuando no se haya encontrado ningún objeto
    if len(cajas) == 0:
        registros.append({
            "archivo": ruta.name,
            "etiqueta": None,
            "confianza": None,
            "x1": None, "y1": None, "x2": None, "y2": None,
        })
        
    # Caso en donde encontramos objetos en las imagenes
    else:
        # Nombres de las categorías
        nombres = [pesos.meta["categories"][i] for i in etiqueta.tolist()]

        # Guardamos en los registros la informacion de cada objeto encontrado
        for nombre, score, box in zip(nombres, confianza.tolist(), cajas.tolist()):
            x1, y1, x2, y2 = box
            registros.append({
                "archivo": ruta.name,
                "etiqueta": nombre,
                "confianza": round(score, 4),
                "x1": round(x1, 1), "y1": round(y1, 1),
                "x2": round(x2, 1), "y2": round(y2, 1),
            })

    # Dibujar las cajas sobre la imagen original
    if len(cajas) > 0:
        nombres = [pesos.meta["categories"][i] for i in etiqueta.tolist()]
        img_anotada = draw_bounding_boxes(
            img,
            boxes=cajas,
            labels=nombres,
            colors="red",
            width=4,
            font_size=30,
        )
    else:
        img_anotada = img

    to_pil_image(img_anotada).save(CARPETA_SALIDA / ruta.name)


#=== Creamos el dt y lo exportamos ===
df = pd.DataFrame(registros)
df.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

print("\n===== RESUMEN =====")
print(f"Total de filas (detecciones): {len(df)}")
print(f"Imágenes procesadas: {df['archivo'].nunique()}")
print(f"\nPrimeras filas:\n{df.head(10)}")
print(f"\nConteo por etiqueta:\n{df['etiqueta'].value_counts()}")
print(f"\nResultados guardados en: {ARCHIVO_CSV}")
print(f"Imágenes anotadas en: {CARPETA_SALIDA}/")

ModuleNotFoundError: No module named 'torchvision'

## YOLO

___

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

#=== Variables Varias ===
CARPETA_ENTRADA = "data/FjordFish/images/train"
CARPETA_SALIDA  = Path("resultados_yolo")
ARCHIVO_CSV     = "detecciones_yolo.csv"
EXTENSION = ".jpg"
UMBRAL_CONFIANZA    = 0.7

os.makedirs(CARPETA_SALIDA, exist_ok=True)

#Cargamos uno de los tantos modelos de yolo que hay ('yolov8s.pt' o 'yolov8m.pt') /////////////////////////////EXPLICAR PQ USAMOS ESE ///////////////////////
model = YOLO("yolov8n.pt") 


carpeta = Path(CARPETA_ENTRADA)
rutas = []

#Misma lógica que en el modelo anterior
for archivo in carpeta.iterdir():
    #Obtenemos solo las imagenes de la carpeta, asi evitamos obtener los archivos ocultoas
    if archivo.suffix == EXTENSION:
        rutas.append(archivo)

def extraer_numero(path):
    numeros = re.findall(r'\d+', path.name)
    if numeros != 0:
        return int(numeros[0])
    else:
        return 0

#Ordenamos en base a los numeros
rutas.sort(key=extraer_numero)
print(f"Se encontraron {len(rutas)} imágenes en '{CARPETA_ENTRADA}'")


#=== procesamos las imagenes y guardamos su información ===
registros = []

for ruta in rutas:
    print(f"Procesando: {ruta.name}")

    # generamos las inferencias con verbose false, para no ver tanto log
    results = model.predict(source=str(ruta), conf=UMBRAL_CONFIANZA, verbose=False)[0]

    # YOLO dibuja y guarda la imagen anotada automáticamente en la carpeta de destino
    results.save(filename=str(CARPETA_SALIDA / ruta.name))

    # Extraer las detecciones obtenidas
    boxes = results.boxes

    # Si no hubo detecciones que superen el umbral
    if len(boxes) == 0:
        registros.append({
            "archivo": ruta.name,
            "etiqueta": None,
            "confianza": None,
            "x1": None, "y1": None, "x2": None, "y2": None,
        })
    else:
        print("Se encontro un objeto")
        # Convertir datos de tensores a arrays de Python/NumPy
        xyxy_list = boxes.xyxy.cpu().numpy()     # Coordenadas [x1, y1, x2, y2]
        conf_list = boxes.conf.cpu().numpy()     # Scores de confianza
        cls_list  = boxes.cls.cpu().numpy()      # IDs numéricos de las clases

        for box, conf, cls_id in zip(xyxy_list, conf_list, cls_list):
            x1, y1, x2, y2 = box
            nombre_clase = model.names[int(cls_id)]  # Traducción de ID a Nombre

            registros.append({
                "archivo": ruta.name,
                "etiqueta": nombre_clase,
                "confianza": round(float(conf), 4),
                "x1": round(float(x1), 1),
                "y1": round(float(y1), 1),
                "x2": round(float(x2), 1),
                "y2": round(float(y2), 1),
            })

# =========================================================
# 5. CREAR DATAFRAME Y EXPORTAR
# =========================================================
df = pd.DataFrame(registros)
df.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8")

print("\n===== RESUMEN YOLO =====")
print(f"Total de filas (detecciones): {len(df)}")
print(f"Imágenes procesadas: {df['archivo'].nunique()}")
print(f"\nPrimeras filas:\n{df.head(10)}")
print(f"\nConteo por etiqueta:\n{df['etiqueta'].value_counts()}")
print(f"\nResultados guardados en: {ARCHIVO_CSV}")
print(f"Imágenes anotadas en: {CARPETA_SALIDA}/")

Se encontraron 2086 imágenes en 'data/FjordFish/images/train'
Procesando: 1.jpg
Procesando: 2.jpg
Procesando: 3.jpg
Procesando: 4.jpg
Procesando: 5.jpg
Procesando: 6.jpg
Procesando: 7.jpg
Procesando: 8.jpg
Procesando: 9.jpg
Procesando: 10.jpg
Procesando: 12.jpg
Procesando: 13.jpg
Procesando: 14.jpg
Procesando: 17.jpg
Procesando: 19.jpg
Procesando: 21.jpg
Procesando: 23.jpg
Procesando: 24.jpg
Procesando: 25.jpg
Procesando: 26.jpg
Procesando: 30.jpg
Procesando: 32.jpg
Procesando: 34.jpg
Procesando: 35.jpg
Procesando: 36.jpg
Procesando: 37.jpg
Procesando: 39.jpg
Procesando: 41.jpg
Procesando: 42.jpg
Procesando: 43.jpg
Procesando: 44.jpg
Procesando: 45.jpg
Procesando: 47.jpg
Procesando: 49.jpg
Procesando: 50.jpg
Procesando: 51.jpg
Procesando: 52.jpg
Procesando: 54.jpg
Procesando: 55.jpg
Procesando: 56.jpg
Procesando: 57.jpg
Procesando: 58.jpg
Procesando: 59.jpg
Procesando: 60.jpg
Procesando: 61.jpg
Procesando: 62.jpg
Procesando: 63.jpg
Procesando: 64.jpg
Procesando: 66.jpg
Procesando: 67.j